In [1]:
import pandas as pd
import numpy as np
import pickle

# Load everything
df_features = pd.read_csv('data_features.csv')
df_predictions = pd.read_csv('model_predictions.csv')
df_clean = pd.read_csv('data_clean.csv')

print('All files loaded')

All files loaded


In [2]:
# Recreate scores and risk bands on full clean dataset
with open('xgb_model.pkl', 'rb') as f:
    xgb = pickle.load(f)

feature_cols = [c for c in df_features.columns if c != 'SeriousDlqin2yrs']

# Get predictions on full dataset
probs = xgb.predict_proba(df_features[feature_cols])[:, 1]

# Convert to credit score
def prob_to_score(prob):
    return int(round(850 - (prob * (850 - 300))))

# Build dashboard dataframe
dashboard_df = df_clean.copy()
dashboard_df = dashboard_df.head(len(probs))  # match lengths
dashboard_df['default_probability'] = probs
dashboard_df['credit_score'] = [prob_to_score(p) for p in probs]

def score_to_band(score):
    if score >= 700:
        return 'Low Risk'
    elif score >= 550:
        return 'Medium Risk'
    else:
        return 'High Risk'

dashboard_df['risk_band'] = dashboard_df['credit_score'].apply(score_to_band)

# Add age group
dashboard_df['age_group'] = pd.cut(
    dashboard_df['age'],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '25-35', '35-45', '45-55', '55-65', '65+']
)

print(dashboard_df.shape)
print(dashboard_df[['age', 'credit_score', 'risk_band', 'SeriousDlqin2yrs']].head(10))

(149377, 15)
   age  credit_score    risk_band  SeriousDlqin2yrs
0   45           577  Medium Risk                 1
1   40           551  Medium Risk                 0
2   38           579  Medium Risk                 0
3   30           810     Low Risk                 0
4   49           560  Medium Risk                 0
5   74           828     Low Risk                 0
6   57           797     Low Risk                 0
7   39           709     Low Risk                 0
8   27           766     Low Risk                 0
9   57           757     Low Risk                 0


In [3]:
# Save dashboard data
dashboard_df.to_csv('dashboard_data.csv', index=False)
print('dashboard_data.csv saved!')
print(f'Total rows: {len(dashboard_df):,}')
print(f'Columns: {list(dashboard_df.columns)}')

dashboard_data.csv saved!
Total rows: 149,377
Columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'age_group', 'default_probability', 'credit_score', 'risk_band']
